In [25]:
!pip install google-auth google-auth-oauthlib google-api-python-client opencv-python imagehash Pillow tqdm

In [29]:
import os

# Check exact folder names
for item in os.listdir("/content/drive/MyDrive"):
    if "dataset" in item.lower() or "DATASET" in item:
        print(repr(item))  # repr shows hidden spaces/characters

'dataset_images'
'dataset_clean'
'DATASET '


In [30]:
# ── Paste this at the top of the script instead ──
import subprocess
subprocess.run(["pip", "install", "imagehash", "opencv-python-headless", "Pillow", "tqdm", "-q"])

from google.colab import drive
drive.mount('/content/drive')

import os

# Find DATASET folder by searching Drive
SOURCE_DIR = None
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for d in dirs:
        if d.upper() == "DATASET":
            SOURCE_DIR = os.path.join(root, d)
            break
    if SOURCE_DIR:
        break

if SOURCE_DIR:
    print(f"✅ Found DATASET at: {SOURCE_DIR}")
else:
    print("❌ Still not found — listing all folders in MyDrive:")
    for item in os.listdir("/content/drive/MyDrive"):
        print(f"  {item}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
❌ Still not found — listing all folders in MyDrive:
  Colab Notebooks
  Classroom
  LAB ICT SEC D
  Muhammad Aslam Khalid_F2024-0671_ICT LAB2
  Untitled document (5).gdoc
  ICT_Lab_4 (1).gslides
  ICT_Lab_4.gslides
  ICT.Lab.04.gdoc
  IEEE FORMAT.docx
  IEEE FORMAT  ICT lab 4.pdf
  IEEE FORMAT (1).pdf
  resume creation.pdf
  resume creation.docx
  Mail Merge .pdf
  Mail MERGE.docx
  mail merge.mdb
  ICT LAB 4_Muhammad Aslam Khalid_F2024-0671_SEC D.pdf
  Untitled document (4).gdoc
  ICT LAB 4_F2024-0671_MUHAMMAD ASLAM KHALID_SEC D.pdf
  ict lab 5.xlsx
  LESCO.xlsx
  LESCO BILL_F2024-0671_MUHAMMAD ASLAM KHALID_SEC D.xlsx
  MUHAMMAD ASLAM KHALID_F2024-0671 - ICT.Lab.05.docx
  MUHAMMAD ASLAM KHALID_F2024-0671_SEC D _ICT ASSIGNMENT 1.pdf
  GG.xlsx
  Ahmer Physics Articles 22-Oct-2024 00-57-47.pdf
  Week 3 ICT LAB_cs secD _F2024-0671.gdoc
  PN DIODE.docx
  MUHAMMAD

In [31]:
"""
Dermalytix — Dataset Cleaner (Google Colab Version)
=====================================================
Run this cell by cell in Google Colab.

STEP 1: Mount Drive
STEP 2: Install dependencies
STEP 3: Run cleaner

Your DATASET folder structure in Drive:
  MyDrive/
    DATASET/
      acne/
      dark spots/
      ...

Output:
  MyDrive/
    DATASET_CLEANED/
      acne/
      dark spots/
      ...
"""

# ─────────────────────────────────────────────
# CELL 1 — Mount Drive + Install dependencies
# ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(["pip", "install", "imagehash", "opencv-python-headless", "Pillow", "tqdm", "-q"])


# ─────────────────────────────────────────────
# CELL 2 — Config (edit these if needed)
# ─────────────────────────────────────────────
import os

# Change this line in CELL 2:
SOURCE_DIR = "/content/drive/MyDrive/DATASET "   # <-- note the trailing space
DEST_DIR   = "/content/drive/MyDrive/DATASET_CLEANED"
BLUR_THRESHOLD  = 100.0   # increase to remove more blurry images
PHASH_THRESHOLD = 5       # 0 = exact dupes only, 5 = near-dupes too
SUPPORTED_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ─────────────────────────────────────────────
# CELL 3 — Run the cleaner
# ─────────────────────────────────────────────
import shutil
import cv2
import imagehash
import numpy as np
from PIL import Image
from tqdm import tqdm

def get_phash(img_path):
    try:
        img = Image.open(img_path).convert("RGB")
        return str(imagehash.phash(img, hash_size=8))
    except:
        return None

def is_blurry(img_path, threshold):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    return score < threshold

def clean_dataset():
    print("\n" + "="*55)
    print("  Dermalytix Dataset Cleaner — Colab Edition")
    print(f"  Source : {SOURCE_DIR}")
    print(f"  Dest   : {DEST_DIR}")
    print(f"  Blur threshold  : {BLUR_THRESHOLD}")
    print(f"  pHash threshold : {PHASH_THRESHOLD}")
    print("="*55 + "\n")

    # Validate source exists
    if not os.path.exists(SOURCE_DIR):
        print(f"❌  SOURCE folder not found: {SOURCE_DIR}")
        print("    Check that your Drive folder is named exactly 'DATASET'")
        return

    # Get condition subfolders
    subfolders = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    if not subfolders:
        print("❌  No subfolders found inside DATASET/")
        return

    print(f"  Found {len(subfolders)} condition folders: {subfolders}\n")

    total_before = 0
    total_kept   = 0
    report       = []

    for condition in subfolders:
        src_folder  = os.path.join(SOURCE_DIR, condition)
        dest_folder = os.path.join(DEST_DIR, condition)
        os.makedirs(dest_folder, exist_ok=True)

        # Get all images
        images = sorted([
            f for f in os.listdir(src_folder)
            if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS
        ])

        before = len(images)
        total_before += before
        print(f"📂  {condition}/ — {before} images")

        seen_hashes = {}
        kept = dupes = blurry_count = 0

        for fname in tqdm(images, desc=f"  Cleaning", leave=False):
            src_path  = os.path.join(src_folder, fname)
            dest_path = os.path.join(dest_folder, fname)

            # 1. Duplicate check via pHash
            h = get_phash(src_path)
            if h is None:
                continue  # unreadable file, skip

            is_dupe = False
            for seen_h in seen_hashes:
                dist = imagehash.hex_to_hash(h) - imagehash.hex_to_hash(seen_h)
                if dist <= PHASH_THRESHOLD:
                    is_dupe = True
                    dupes += 1
                    break

            if is_dupe:
                continue

            seen_hashes[h] = fname

            # 2. Blur check
            if is_blurry(src_path, BLUR_THRESHOLD):
                blurry_count += 1
                continue

            # 3. Copy clean image to DATASET_CLEANED
            shutil.copy2(src_path, dest_path)
            kept += 1

        total_kept += kept
        report.append({
            "class":   condition,
            "before":  before,
            "dupes":   dupes,
            "blurry":  blurry_count,
            "kept":    kept,
        })
        print(f"  ✓ Kept {kept} | Removed {dupes + blurry_count} "
              f"(dupes: {dupes}, blurry: {blurry_count})\n")

    # ── Summary table ──
    removed = total_before - total_kept
    print("="*58)
    print("  SUMMARY REPORT")
    print("="*58)
    print(f"  {'Condition':<22} {'Before':>6} {'Dupes':>6} {'Blurry':>7} {'Kept':>6}")
    print(f"  {'-'*54}")
    for r in report:
        status = "⚠️  LOW" if r["kept"] < 500 else "✅"
        print(f"  {r['class']:<22} {r['before']:>6} {r['dupes']:>6} "
              f"{r['blurry']:>7} {r['kept']:>6}  {status}")
    print(f"  {'-'*54}")
    print(f"  {'TOTAL':<22} {total_before:>6} {'':>6} {'':>7} {total_kept:>6}")
    print(f"\n  Removed : {removed} images ({100*removed/max(total_before,1):.1f}%)")
    print(f"  Kept    : {total_kept} images ({100*total_kept/max(total_before,1):.1f}%)")
    print(f"\n  ✅  Clean dataset saved to Drive → DATASET_CLEANED/")
    print(f"  ⚠️   Classes marked LOW need augmentation (under 500 images)")
    print("="*58 + "\n")

clean_dataset()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  Dermalytix Dataset Cleaner — Colab Edition
  Source : /content/drive/MyDrive/DATASET 
  Dest   : /content/drive/MyDrive/DATASET_CLEANED
  Blur threshold  : 100.0
  pHash threshold : 5

  Found 7 condition folders: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']

📂  Acne/ — 2035 images


  ✓ Kept 1264 | Removed 771 (dupes: 134, blurry: 637)

📂  Dark circles/ — 300 images


  ✓ Kept 209 | Removed 91 (dupes: 66, blurry: 25)

📂  dark spots/ — 503 images


  ✓ Kept 294 | Removed 209 (dupes: 70, blurry: 139)

📂  dry/ — 758 images


  ✓ Kept 710 | Removed 48 (dupes: 17, blurry: 31)

📂  normal/ — 170 images


  ✓ Kept 152 | Removed 18 (dupes: 6, blurry: 12)

📂  oily/ — 120 images


  ✓ Kept 110 | Removed 10 (dupes: 0, blurry: 10)

📂  wrinkles:FINE LINES/ — 500 images


  ✓ Kept 396 | Removed 104 (dupes: 84, blurry: 20)

  SUMMARY REPORT
  Condition              Before  Dupes  Blurry   Kept
  ------------------------------------------------------
  Acne                     2035    134     637   1264  ✅
  Dark circles              300     66      25    209  ⚠️  LOW
  dark spots                503     70     139    294  ⚠️  LOW
  dry                       758     17      31    710  ✅
  normal                    170      6      12    152  ⚠️  LOW
  oily                      120      0      10    110  ⚠️  LOW
  wrinkles:FINE LINES       500     84      20    396  ⚠️  LOW
  ------------------------------------------------------
  TOTAL                    4386                  3135

  Removed : 1251 images (28.5%)
  Kept    : 3135 images (71.5%)

  ✅  Clean dataset saved to Drive → DATASET_CLEANED/
  ⚠️   Classes marked LOW need augmentation (under 500 images)



In [32]:
"""
Dermalytix — Dataset Augmentation (Colab Version)
===================================================
Reads from DATASET_CLEANED, augments LOW classes to 800 images,
saves result to DATASET_AUGMENTED on Drive.

Current status:
  Acne          1264  ✅ (will keep as-is, cap at 800 for balance)
  Dry            710  ✅ (will keep as-is)
  Wrinkles       396  ⚠️ needs +404
  Dark Spots     294  ⚠️ needs +506
  Dark Circles   209  ⚠️ needs +591
  Normal         152  ⚠️ needs +648
  Oily           110  ⚠️ needs +690

Target: 800 images per class
"""

# ── CELL 1: Install & import ──────────────────────────────
import subprocess
subprocess.run(["pip", "install", "albumentations", "opencv-python-headless",
                "Pillow", "tqdm", "-q"])

import os
import cv2
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
import albumentations as A

# ── CELL 2: Config ────────────────────────────────────────
SOURCE_DIR = "/content/drive/MyDrive/DATASET_CLEANED"
DEST_DIR   = "/content/drive/MyDrive/DATASET_AUGMENTED"
TARGET     = 800          # target images per class
SEED       = 42
random.seed(SEED)
np.random.seed(SEED)

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── CELL 3: Augmentation pipeline ─────────────────────────
# Skin-safe transforms — no extreme distortions
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(
        shift_limit=0.05, scale_limit=0.1,
        rotate_limit=15, p=0.5
    ),
    A.OneOf([
        A.RandomBrightnessContrast(
            brightness_limit=0.2, contrast_limit=0.2, p=1.0
        ),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20,
            val_shift_limit=10, p=1.0
        ),
    ], p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(p=1.0),
        A.Sharpen(p=1.0),
    ], p=0.3),
    A.RandomResizedCrop(
        size=(224, 224), scale=(0.85, 1.0), p=0.4
    ),
    A.Resize(224, 224),   # ensure all outputs are 224x224
])

# ── CELL 4: Run augmentation ──────────────────────────────
def load_images(folder):
    paths = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS
    ]
    return paths

def augment_class(condition, src_folder, dest_folder, target):
    images = load_images(src_folder)
    current = len(images)
    os.makedirs(dest_folder, exist_ok=True)

    # Copy all originals first
    copied = 0
    for path in images:
        fname = os.path.basename(path)
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.resize(img, (224, 224))
        cv2.imwrite(os.path.join(dest_folder, fname), img)
        copied += 1

    if current >= target:
        print(f"  ✅ {condition}: {current} images — no augmentation needed (copied {copied})")
        return copied

    # Generate augmented images to reach target
    needed   = target - copied
    aug_count = 0

    print(f"  🔄 {condition}: {current} → generating {needed} augmented images...")

    with tqdm(total=needed, desc=f"  {condition}", leave=False) as pbar:
        while aug_count < needed:
            # Pick a random original image
            src_path = random.choice(images)
            img = cv2.imread(src_path)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Apply augmentation
            augmented = augment(image=img_rgb)["image"]
            aug_bgr   = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)

            # Save with aug_ prefix + counter
            out_name = f"aug_{aug_count:05d}.jpg"
            cv2.imwrite(os.path.join(dest_folder, out_name), aug_bgr)
            aug_count += 1
            pbar.update(1)

    total = copied + aug_count
    print(f"  ✅ {condition}: {current} originals + {aug_count} augmented = {total} total")
    return total


def run():
    print("\n" + "="*58)
    print("  Dermalytix Dataset Augmentation")
    print(f"  Source  : {SOURCE_DIR}")
    print(f"  Dest    : {DEST_DIR}")
    print(f"  Target  : {TARGET} images per class")
    print("="*58 + "\n")

    if not os.path.exists(SOURCE_DIR):
        print(f"❌ Source not found: {SOURCE_DIR}")
        return

    conditions = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    print(f"  Found {len(conditions)} classes: {conditions}\n")

    report = []
    for condition in conditions:
        src = os.path.join(SOURCE_DIR, condition)
        dst = os.path.join(DEST_DIR, condition)
        total = augment_class(condition, src, dst, TARGET)
        report.append({"class": condition, "final": total})

    # Summary
    print("\n" + "="*58)
    print("  AUGMENTATION COMPLETE — FINAL CLASS COUNTS")
    print("="*58)
    print(f"  {'Class':<22} {'Images':>8}  {'Status':>10}")
    print(f"  {'-'*50}")
    for r in report:
        status = "✅ Ready" if r["final"] >= TARGET else "⚠️ Low"
        print(f"  {r['class']:<22} {r['final']:>8}  {status:>10}")
    total_imgs = sum(r["final"] for r in report)
    print(f"  {'-'*50}")
    print(f"  {'TOTAL':<22} {total_imgs:>8}")
    print(f"\n  ✅ Augmented dataset saved to Drive → DATASET_AUGMENTED/")
    print(f"  📌 Next step: run train/val/test split on DATASET_AUGMENTED")
    print("="*58 + "\n")

run()

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)



  Dermalytix Dataset Augmentation
  Source  : /content/drive/MyDrive/DATASET_CLEANED
  Dest    : /content/drive/MyDrive/DATASET_AUGMENTED
  Target  : 800 images per class

  Found 7 classes: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']

  ✅ Acne: 1264 images — no augmentation needed (copied 1264)
  🔄 Dark circles: 209 → generating 591 augmented images...


  ✅ Dark circles: 209 originals + 591 augmented = 800 total
  🔄 dark spots: 294 → generating 506 augmented images...


  ✅ dark spots: 294 originals + 506 augmented = 800 total
  🔄 dry: 710 → generating 90 augmented images...


  ✅ dry: 710 originals + 90 augmented = 800 total
  🔄 normal: 152 → generating 648 augmented images...


  ✅ normal: 152 originals + 648 augmented = 800 total
  🔄 oily: 110 → generating 690 augmented images...


  ✅ oily: 110 originals + 690 augmented = 800 total
  🔄 wrinkles:FINE LINES: 396 → generating 404 augmented images...


  ✅ wrinkles:FINE LINES: 396 originals + 404 augmented = 800 total

  AUGMENTATION COMPLETE — FINAL CLASS COUNTS
  Class                    Images      Status
  --------------------------------------------------
  Acne                       1264     ✅ Ready
  Dark circles                800     ✅ Ready
  dark spots                  800     ✅ Ready
  dry                         800     ✅ Ready
  normal                      800     ✅ Ready
  oily                        800     ✅ Ready
  wrinkles:FINE LINES         800     ✅ Ready
  --------------------------------------------------
  TOTAL                      6064

  ✅ Augmented dataset saved to Drive → DATASET_AUGMENTED/
  📌 Next step: run train/val/test split on DATASET_AUGMENTED



In [36]:
"""
Dermalytix — Train / Val / Test Split (Colab Version)
=======================================================
Reads from DATASET_AUGMENTED, splits into 70/15/15,
saves to DATASET_SPLIT on Drive.

Output structure:
  DATASET_SPLIT/
    train/
      acne/         → 70% of images
      dark circles/
      ...
    val/
      acne/         → 15% of images
      ...
    test/
      acne/         → 15% of images (real images only, no aug_)
      ...

IMPORTANT RULES:
  - val and test sets contain REAL images only (no aug_ files)
  - train set gets all augmented images + remaining real ones
  - Splits are stratified — same ratio per class
"""

import os
import shutil
import random
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────
SOURCE_DIR  = "/content/drive/MyDrive/DATASET_AUGMENTED"
DEST_DIR    = "/content/drive/MyDrive/DATASET_SPLIT"
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SEED        = 42
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

random.seed(SEED)

# ── Run split ─────────────────────────────────────────────
def split_dataset():
    print("\n" + "="*58)
    print("  Dermalytix Train / Val / Test Split")
    print(f"  Source : {SOURCE_DIR}")
    print(f"  Dest   : {DEST_DIR}")
    print(f"  Split  : {int(TRAIN_RATIO*100)}% train / "
          f"{int(VAL_RATIO*100)}% val / {int(TEST_RATIO*100)}% test")
    print(f"  Rule   : val + test = real images only (no aug_)")
    print("="*58 + "\n")

    if not os.path.exists(SOURCE_DIR):
        print(f"❌ Not found: {SOURCE_DIR}")
        return

    conditions = sorted([
        d for d in os.listdir(SOURCE_DIR)
        if os.path.isdir(os.path.join(SOURCE_DIR, d))
    ])

    print(f"  Found {len(conditions)} classes: {conditions}\n")

    report = []

    for condition in conditions:
        src_folder = os.path.join(SOURCE_DIR, condition)

        all_files = sorted([
            f for f in os.listdir(src_folder)
            if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS
        ])

        # Separate real vs augmented
        real_imgs = [f for f in all_files if not f.startswith("aug_")]
        aug_imgs  = [f for f in all_files if f.startswith("aug_")]

        # Shuffle real images for random split
        random.shuffle(real_imgs)

        # Val and test come from REAL images only
        n_real    = len(real_imgs)
        n_val     = max(1, round(n_real * VAL_RATIO))
        n_test    = max(1, round(n_real * TEST_RATIO))
        n_train_real = n_real - n_val - n_test

        val_files  = real_imgs[:n_val]
        test_files = real_imgs[n_val:n_val + n_test]
        train_real = real_imgs[n_val + n_test:]

        # Train = remaining real + ALL augmented
        train_files = train_real + aug_imgs

        splits = {
            "train": train_files,
            "val":   val_files,
            "test":  test_files,
        }

        for split_name, files in splits.items():
            dest_folder = os.path.join(DEST_DIR, split_name, condition)
            os.makedirs(dest_folder, exist_ok=True)
            for fname in tqdm(files, desc=f"  {split_name}/{condition}", leave=False):
                src  = os.path.join(src_folder, fname)
                dest = os.path.join(dest_folder, fname)
                shutil.copy2(src, dest)

        report.append({
            "class":  condition,
            "train":  len(train_files),
            "val":    len(val_files),
            "test":   len(test_files),
            "total":  len(all_files),
        })
        print(f"  ✅ {condition:<22} "
              f"train={len(train_files):>4}  "
              f"val={len(val_files):>3}  "
              f"test={len(test_files):>3}")

    # ── Summary ──
    total_train = sum(r["train"] for r in report)
    total_val   = sum(r["val"]   for r in report)
    total_test  = sum(r["test"]  for r in report)
    grand_total = total_train + total_val + total_test

    print("\n" + "="*58)
    print("  SPLIT SUMMARY")
    print("="*58)
    print(f"  {'Class':<22} {'Train':>6} {'Val':>5} {'Test':>5} {'Total':>7}")
    print(f"  {'-'*52}")
    for r in report:
        print(f"  {r['class']:<22} {r['train']:>6} {r['val']:>5} "
              f"{r['test']:>5} {r['total']:>7}")
    print(f"  {'-'*52}")
    print(f"  {'TOTAL':<22} {total_train:>6} {total_val:>5} "
          f"{total_test:>5} {grand_total:>7}")
    print(f"\n  ✅  Dataset split saved to Drive → DATASET_SPLIT/")
    print(f"  📌  Next step: train EfficientNetB0 on DATASET_SPLIT/train")
    print("="*58 + "\n")

split_dataset()


  Dermalytix Train / Val / Test Split
  Source : /content/drive/MyDrive/DATASET_AUGMENTED
  Dest   : /content/drive/MyDrive/DATASET_SPLIT
  Split  : 70% train / 15% val / 15% test
  Rule   : val + test = real images only (no aug_)

  Found 7 classes: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']



  ✅ Acne                   train= 884  val=190  test=190


  ✅ Dark circles           train= 738  val= 31  test= 31


  ✅ dark spots             train= 712  val= 44  test= 44


  ✅ dry                    train= 588  val=106  test=106


  ✅ normal                 train= 754  val= 23  test= 23


  ✅ oily                   train= 768  val= 16  test= 16


  ✅ wrinkles:FINE LINES    train= 682  val= 59  test= 59

  SPLIT SUMMARY
  Class                   Train   Val  Test   Total
  ----------------------------------------------------
  Acne                      884   190   190    1264
  Dark circles              738    31    31     800
  dark spots                712    44    44     800
  dry                       588   106   106     800
  normal                    754    23    23     800
  oily                      768    16    16     800
  wrinkles:FINE LINES       682    59    59     800
  ----------------------------------------------------
  TOTAL                    5126   469   469    6064

  ✅  Dataset split saved to Drive → DATASET_SPLIT/
  📌  Next step: train EfficientNetB0 on DATASET_SPLIT/train



In [37]:
import os, shutil, random
from tqdm import tqdm

SOURCE_DIR = "/content/drive/MyDrive/DATASET_AUGMENTED"
DEST_DIR   = "/content/drive/MyDrive/DATASET_SPLIT"
SEED       = 42
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
random.seed(SEED)

# Only process conditions that are MISSING or INCOMPLETE
for split in ["train", "val", "test"]:
    split_path = os.path.join(DEST_DIR, split)
    existing = os.listdir(split_path) if os.path.exists(split_path) else []
    print(f"{split}/ has: {existing}")

train/ has: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']
val/ has: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']
test/ has: ['Acne', 'Dark circles', 'dark spots', 'dry', 'normal', 'oily', 'wrinkles:FINE LINES']


In [38]:
import os

split_dir = "/content/drive/MyDrive/DATASET_SPLIT"

for split in ["train", "val", "test"]:
    path = os.path.join(split_dir, split)
    conditions = os.listdir(path)
    total = sum(len(os.listdir(os.path.join(path, c))) for c in conditions)
    print(f"\n{split}/ → {len(conditions)} folders, {total} images")
    for c in sorted(conditions):
        count = len(os.listdir(os.path.join(path, c)))
        print(f"  {c:<25} {count}")


train/ → 7 folders, 5126 images
  Acne                      884
  Dark circles              738
  dark spots                712
  dry                       588
  normal                    754
  oily                      768
  wrinkles:FINE LINES       682

val/ → 7 folders, 469 images
  Acne                      190
  Dark circles              31
  dark spots                44
  dry                       106
  normal                    23
  oily                      16
  wrinkles:FINE LINES       59

test/ → 7 folders, 469 images
  Acne                      190
  Dark circles              31
  dark spots                44
  dry                       106
  normal                    23
  oily                      16
  wrinkles:FINE LINES       59


In [34]:
import os

# Check what's actually inside DATASET_AUGMENTED
source = "/content/drive/MyDrive/DATASET_AUGMENTED"

print("Does DATASET_AUGMENTED exist?", os.path.exists(source))
print("\nFolders inside it:")
for item in os.listdir(source):
    full = os.path.join(source, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {repr(item)} → {count} files")

# Also check DATASET_SPLIT
split = "/content/drive/MyDrive/DATASET_SPLIT"
print("\n\nDATASET_SPLIT exists?", os.path.exists(split))
if os.path.exists(split):
    for split_name in ["train", "val", "test"]:
        sp = os.path.join(split, split_name)
        print(f"\n  {split_name}/ exists?", os.path.exists(sp))
        if os.path.exists(sp):
            for cond in os.listdir(sp):
                cp = os.path.join(sp, cond)
                print(f"    {repr(cond)} → {len(os.listdir(cp))} files")

Does DATASET_AUGMENTED exist? True

Folders inside it:
  'Acne' → 1264 files
  'Dark circles' → 800 files
  'dark spots' → 800 files
  'dry' → 800 files
  'normal' → 800 files
  'oily' → 800 files
  'wrinkles:FINE LINES' → 800 files


DATASET_SPLIT exists? True

  train/ exists? True
    'Acne' → 884 files
    'Dark circles' → 738 files
    'dark spots' → 712 files
    'dry' → 588 files
    'normal' → 754 files
    'oily' → 768 files
    'wrinkles:FINE LINES' → 682 files

  val/ exists? True
    'Acne' → 190 files
    'Dark circles' → 31 files
    'dark spots' → 44 files
    'dry' → 106 files
    'normal' → 23 files
    'oily' → 16 files
    'wrinkles:FINE LINES' → 59 files

  test/ exists? True
    'Acne' → 190 files
    'Dark circles' → 31 files
    'dark spots' → 44 files
    'dry' → 106 files
    'normal' → 23 files
    'oily' → 16 files
    'wrinkles:FINE LINES' → 59 files
